<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

# 🧩 把 Subagent 当工具用

**一句话定位**：多 agent 系统最简单的一种搭法——把一个专精某件事的 agent 整个包成一个 `@tool`，让「主 agent」像调用普通函数一样调用它。



</div>

这种模式叫 **orchestrator-worker**（或 supervisor 模式）：主 agent 负责理解用户意图、决定调用谁；subagent 只负责把自己那一小块任务做好，不需要知道整体上下文。

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## 🔨 创建 subagent

每个 subagent 就是一个普通的 `create_agent(...)`，只是工具集更窄、职责更单一——这里一个只会算平方根，一个只会算平方。

In [2]:
from langchain.tools import tool


@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x**0.5


@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x**2

In [3]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[square]
)

## 📞 调用 subagent

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**易踩坑**：想把 subagent 接到主 agent 上，不能直接把 subagent 对象塞进 `tools=[...]`——必须再包一层 `@tool` 函数，在函数体内手动 `subagent.invoke(...)`，再把结果 `return` 出来。下面这个 `call_subagent_1` 示例里 `return` 后面是空的，是故意留给你补全的：记得把 `subagent_1.invoke(...)` 返回的 `response` 里最后一条消息的内容 `return` 出去，否则主 agent 永远收不到 subagent 的计算结果。

</div>

In [4]:
from langchain.messages import HumanMessage


@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square root of {x}")]}
    )
    return response["messages"][-1].content


@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke(
        {"messages": [HumanMessage(content=f"Calculate the square of {x}")]}
    )
    return response["messages"][-1].content


## Creating the main agent

main_agent = create_agent(
    model="gpt-5-nano",
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.",
)

## ✅ 测试

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：Subagent-as-tool 模式的本质是「把一个 agent 的全部能力压缩成一次函数调用」——对主 agent 来说，调用 subagent 和调用普通工具没有区别，复杂度被封装起来了。

</div>

In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='6dd6ed3c-b3da-400b-aa50-75b622cc844c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 410, 'prompt_tokens': 202, 'total_tokens': 612, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZpjlP8fffel8jJKKcsQ22R89EI6K', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd73f-15f3-7bb0-9643-e59b87fc294c-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_ltOrECMDy3ZDiRRYqOAs6ZtI', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 202, 'o